In [1]:
!git clone --depth 1 https://github.com/ecri-data/ECRI-Dataset.git /content/ecri

Cloning into '/content/ecri'...
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 17 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (17/17), 1.32 MiB | 9.81 MiB/s, done.


In [2]:
!mkdir -p data && cp $(find /content/ecri -name "ECRIDataSet.csv") $(find /content/ecri -name "ECRIRawData.csv") data/

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!mkdir -p /content/drive/MyDrive/ecri_outputs results && ln -s /content/drive/MyDrive/ecri_outputs results/neural_network

In [6]:
!python notebooks/Supriya_Neural_network_implementation.py --dataset data/ECRIDataSet.csv --raw-dataset data/ECRIRawData.csv --output-dir results/neural_network/smoke --models neural --group school --target RMwidRS_target --epochs 5 --pretrain-epochs 5 --no-tsne

Raw-data check: {'dataset_students': 6803, 'rebuilt_students': 6916, 'shared_students': 6803, 'dataset_students_missing_from_raw': 0, 'cells_compared': 115651, 'mismatched_cells': 0, 'verified': True, 'labels': {'label_definition': 'gain(t=1 minus t=0) > mean gain of Tx=0 students', 'RMwidRS_target': {'released_labels': 946, 'rebuilt_and_compared': 946, 'mismatched_labels': 0, 'control_mean_gain': 26.321083172147002}, 'RMwdaRS_target': {'released_labels': 946, 'rebuilt_and_compared': 946, 'mismatched_labels': 0, 'control_mean_gain': 10.754352030947775}, 'verified': True}}
Dataset: 6803 student rows; pretraining pool: 6916 students; device: cpu; targets: ('RMwidRS_target',); evaluating all rows; pretraining loss: cosine; models: ['MLP_zero', 'MLP_mean', 'MLP_indicator', 'MaskMLP']
school | RMwidRS_target | fold 1 | MLP_zero | AP=0.6988
school | RMwidRS_target | fold 1 | MLP_mean | AP=0.7101
school | RMwidRS_target | fold 1 | MLP_indicator | AP=0.6379
school | RMwidRS_target | fold 1 | M

In [7]:
!python notebooks/Supriya_Neural_network_implementation.py --dataset data/ECRIDataSet.csv --raw-dataset data/ECRIRawData.csv --output-dir results/neural_network/main --importance --loss-ablation

Raw-data check: {'dataset_students': 6803, 'rebuilt_students': 6916, 'shared_students': 6803, 'dataset_students_missing_from_raw': 0, 'cells_compared': 115651, 'mismatched_cells': 0, 'verified': True, 'labels': {'label_definition': 'gain(t=1 minus t=0) > mean gain of Tx=0 students', 'RMwidRS_target': {'released_labels': 946, 'rebuilt_and_compared': 946, 'mismatched_labels': 0, 'control_mean_gain': 26.321083172147002}, 'RMwdaRS_target': {'released_labels': 946, 'rebuilt_and_compared': 946, 'mismatched_labels': 0, 'control_mean_gain': 10.754352030947775}, 'verified': True}}
Dataset: 6803 student rows; pretraining pool: 6916 students; device: cpu; targets: ['RMwidRS_target', 'RMwdaRS_target']; evaluating all rows; pretraining loss: cosine; models: ['LogisticRegression', 'XGBoost', 'MLP_zero', 'MLP_mean', 'MLP_indicator', 'MaskMLP']
school | RMwidRS_target | fold 1 | LogisticRegression | AP=0.7321
school | RMwidRS_target | fold 1 | XGBoost | AP=0.7366
school | RMwidRS_target | fold 1 | MLP

In [8]:
!rm -rf results/neural_network/smoke
!python notebooks/Supriya_Neural_network_report.py --outputs results/neural_network

Runs found: main
NOTE: robustness needs a setting with at least two seeds; none found.
NOTE: nothing to write for robustness.csv
NOTE: nothing to write for maskmlp_wins.csv
Report written to results/neural_network/report (primary setting: loss=cosine, pretrain=all, rows=all)


In [11]:
%%bash
export OMP_NUM_THREADS=1 MKL_NUM_THREADS=1
BASE="--dataset data/ECRIDataSet.csv --raw-dataset data/ECRIRawData.csv --no-tsne"
run() {
  name=$1; shift
  if [ -f results/neural_network/$name/results.csv ]; then echo "skip $name"; return; fi
  python notebooks/Supriya_Neural_network_implementation.py $BASE --output-dir results/neural_network/$name "$@" \
    > log_${name//\//_}.txt 2>&1 && echo "done $name" || echo "FAILED $name"
}
throttle() { while (( $(jobs -rp | wc -l) >= 2 )); do wait -n; done; }

for s in 43 44 45 46;    do throttle; run seeds/s$s --seed $s & done
for s in 42 43 44 45 46; do throttle; run train/s$s --pretrain-scope train --seed $s & done
for s in 42 43 44;       do throttle; run complete/all-s$s --eval-rows complete --seed $s & done
for s in 42 43 44;       do throttle; run complete/train-s$s --eval-rows complete --pretrain-scope train --seed $s & done
for s in 42 43 44;       do throttle; run simsiam/s$s --pretrain-loss simsiam --seed $s & done
for s in 42 43 44;       do throttle; run contrastive/s$s --pretrain-loss contrastive --seed $s & done
wait
echo "ALL FINISHED"

done seeds/s43
done seeds/s44
done seeds/s45
done seeds/s46
done train/s42
done train/s43
done train/s44
done train/s45
done complete/all-s42
done complete/all-s43
done train/s46
done complete/all-s44
done complete/train-s42
done complete/train-s43
done simsiam/s42
done complete/train-s44
done simsiam/s43
done simsiam/s44
done contrastive/s42
done contrastive/s43
done contrastive/s44
ALL FINISHED


In [12]:
!python notebooks/Supriya_Neural_network_report.py --outputs results/neural_network
!zip -qr results.zip results/neural_network/

Runs found: complete/all-s42, complete/all-s43, complete/all-s44, complete/train-s42, complete/train-s43, complete/train-s44, contrastive/s42, contrastive/s43, contrastive/s44, main, seeds/s43, seeds/s44, seeds/s45, seeds/s46, simsiam/s42, simsiam/s43, simsiam/s44, train/s42, train/s43, train/s44, train/s45, train/s46
Report written to results/neural_network/report (primary setting: loss=cosine, pretrain=all, rows=all)


In [13]:
from google.colab import files; files.download("results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>